In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import tables
import tqdm.notebook as tqdm

In [2]:
model = AutoModelForSeq2SeqLM.from_pretrained("./models/nllb_teacher").cuda().eval()
tokenizer = AutoTokenizer.from_pretrained("./models/nllb_teacher")

Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

In [3]:
with tables.open_file("./data/tokenized_dataset.hdf5", mode="r") as file:
    source_ids = file.root.source_ids.read()
    target_ids = file.root.target_ids.read()
    source_mask = file.root.source_mask.read()
    target_mask = file.root.target_mask.read()

In [ ]:
batch_size = 4096

filters = tables.Filters(3, "blosc:lz4")

with tables.open_file("./data/tokenized_dataset_synth.hdf5", "w") as file, torch.no_grad():
    new_source_ids = file.create_carray(file.root, "source_ids", tables.Int32Atom(), shape=source_ids.shape, filters=filters)
    new_target_ids = file.create_carray(file.root, "target_ids", tables.Int32Atom(), shape=source_ids.shape, filters=filters)
    new_source_mask = file.create_carray(file.root, "source_mask", tables.BoolAtom(), shape=source_ids.shape, filters=filters)
    new_target_mask = file.create_carray(file.root, "target_mask", tables.BoolAtom(), shape=source_ids.shape, filters=filters)
    synth_ids = file.create_carray(file.root, "synth_ids", tables.Int32Atom(), shape=source_ids.shape, filters=filters)
    synth_mask = file.create_carray(file.root, "synth_mask", tables.BoolAtom(), shape=source_ids.shape, filters=filters)

    for i in tqdm.trange(0, source_ids.shape[0], batch_size):
        ids_tensor = torch.from_numpy(source_ids[i:i + batch_size]).cuda()
        mask_tensor = torch.from_numpy(source_mask[i:i + batch_size]).cuda()
        outputs = model.generate(input_ids=ids_tensor, attention_mask=mask_tensor, max_length=64, forced_bos_token_id=tokenizer.convert_tokens_to_ids("rus_Cyrl"))
        if outputs.size(1) < 64:
            outputs = torch.concat([outputs, torch.full(size=(outputs.size(0), 64 - outputs.size(1)), fill_value=tokenizer.pad_token_id, dtype=outputs.dtype, device="cuda")], dim=1)
        mask = outputs.ne(tokenizer.pad_token_id)
        

        new_source_ids[i:i + batch_size] = source_ids[i:i + batch_size]
        new_target_ids[i:i + batch_size] = target_ids[i:i + batch_size]
        new_source_mask[i:i + batch_size] = source_mask[i:i + batch_size]
        new_target_mask[i:i + batch_size] = target_mask[i:i + batch_size]
        synth_ids[i:i + batch_size] = outputs.cpu().numpy()[...]
        synth_mask[i:i + batch_size] = mask.cpu().numpy()[...]

        del outputs
        del mask

  0%|          | 0/756 [00:00<?, ?it/s]

In [ ]:
from extra import upload
upload("./data/tokenized_dataset_synth.hdf5", "data")